









# Projet Python — Groupe 1
**Solutions des exercices** (structures de données, récursivité, classes)

Exécuter les cellules dans l'ordre (`Shift+Entrée`) ou avec **Run All**.

---
# Partie A — Structures de données
## A.1 — Fibonacci avec mise en cache

In [1]:
def fibonacci(n, cache=None):
    if cache is None:
        cache = {0: 0, 1: 1}
    if n not in cache:
        cache[n] = fibonacci(n - 1, cache) + fibonacci(n - 2, cache)
    return cache[n]

print(fibonacci(10))   # 55
assert fibonacci(10) == 55
print(fibonacci(100))  # grand n : instantané grâce au cache

55
354224848179261915075


## A.2 — Anagrammes

In [2]:
def sont_anagrammes(a, b):
    def norm(s):
        return sorted(s.lower().replace(' ', ''))
    return norm(a) == norm(b)

print(sont_anagrammes("chien", "niche"))   # True
print(sont_anagrammes("Marie", "Aimer"))   # True (majuscules ignorées)
print(sont_anagrammes("chat", "chien"))    # False

assert sont_anagrammes("chien", "niche")
assert not sont_anagrammes("chat", "chien")

True
True
False


---
# Partie B — Récursivité
## B.1 — Aplatir une liste

In [3]:
def aplatir(liste):
    res = []
    for e in liste:
        if isinstance(e, list):
            res.extend(aplatir(e))   # cas récursif : sous-liste
        else:
            res.append(e)            # cas de base : élément simple
    return res

print(aplatir([1, [2, [3, 4]], 5]))   # [1, 2, 3, 4, 5]
assert aplatir([1, [2, [3, 4]], 5]) == [1, 2, 3, 4, 5]
assert aplatir([]) == []

[1, 2, 3, 4, 5]


## B.2 — Puissance rapide

In [4]:
def puissance(x, n):
    if n == 0:
        return 1
    m = puissance(x, n // 2)
    r = m * m
    return r * x if n % 2 else r

print(puissance(2, 10))   # 1024
assert puissance(2, 10) == 1024
assert all(puissance(3, k) == 3 ** k for k in range(20))

1024


**Combien de multiplications, comparé à la version simple ?** On les compte :

In [5]:
def puissance_simple_comptee(x, n):
    res, nb = 1, 0
    for _ in range(n):
        res *= x
        nb += 1
    return res, nb

def puissance_rapide_comptee(x, n):
    if n == 0:
        return 1, 0
    m, nb = puissance_rapide_comptee(x, n // 2)
    r = m * m
    nb += 1
    if n % 2:
        r *= x
        nb += 1
    return r, nb

print(f"{'n':>10} | {'version simple':>14} | {'version rapide':>14}")
print("-" * 46)
for n in [10, 1_000, 1_000_000]:
    _, nb_simple = puissance_simple_comptee(1, n)
    _, nb_rapide = puissance_rapide_comptee(1, n)
    print(f"{n:>10} | {nb_simple:>14} | {nb_rapide:>14}")

         n | version simple | version rapide
----------------------------------------------
        10 |             10 |              6
      1000 |           1000 |             16
   1000000 |        1000000 |             27


**Complexité :** version simple en **O(n)** (n multiplications) ; version rapide en **O(log n)** (on divise n par 2 à chaque appel, soit environ log₂(n) appels).

---
# Partie C — Classes
## C.1 — Attribut de classe : entier ou liste

In [6]:
# Cas 1 : attribut de classe de type ENTIER
class Compteur:
    valeur = 0

c1, c2 = Compteur(), Compteur()
c1.valeur += 1

print("c1.valeur :", c1.valeur)             # 1
print("c2.valeur :", c2.valeur)             # 0 -> inchangé
print("Compteur.valeur :", Compteur.valeur) # 0
print("c1 a maintenant son propre attribut :", "valeur" in c1.__dict__)

c1.valeur : 1
c2.valeur : 0
Compteur.valeur : 0
c1 a maintenant son propre attribut : True


In [7]:
# Cas 2 : attribut de classe de type LISTE
class Panier:
    articles = []

p1, p2 = Panier(), Panier()
p1.articles.append("pomme")

print("p1.articles :", p1.articles)      # ['pomme']
print("p2.articles :", p2.articles)      # ['pomme'] -> partagé !
print("même objet  :", p1.articles is p2.articles)

p1.articles : ['pomme']
p2.articles : ['pomme']
même objet  : True


**Différence entre les deux cas :**
- **Entier (immuable)** : `c1.valeur += 1` équivaut à `c1.valeur = c1.valeur + 1`. C'est une *réaffectation* : Python crée un attribut d'instance sur `c1` qui masque l'attribut de classe. `c2` ne voit rien.
- **Liste (mutable)** : `append` *modifie l'objet en place*, sans réaffectation. Il n'existe qu'une seule liste, partagée par toutes les instances.

**Pourquoi faire attention avec un attribut de classe mutable ?** Parce que tous les objets partagent le même objet : une modification faite par l'un est visible chez tous les autres. Exemple concret :

In [8]:
# Le piège : les notes d'Alice apparaissent chez Bob
class EtudiantPiege:
    notes = []                       # attribut de classe mutable : DANGER

    def __init__(self, nom):
        self.nom = nom

alice = EtudiantPiege("Alice")
bob = EtudiantPiege("Bob")
alice.notes.append(18)
print("Notes de Bob :", bob.notes)   # [18] alors que Bob n'a rien reçu !

Notes de Bob : [18]


In [9]:
# La solution : un attribut d'instance, créé dans __init__
class Etudiant:
    def __init__(self, nom):
        self.nom = nom
        self.notes = []              # propre à chaque objet

alice = Etudiant("Alice")
bob = Etudiant("Bob")
alice.notes.append(18)
print("Notes d'Alice :", alice.notes)   # [18]
print("Notes de Bob  :", bob.notes)     # []

Notes d'Alice : [18]
Notes de Bob  : []
